# Fase 1 · Recolección del corpus científico (arXiv)

Este notebook construye `data/scientific_raw_corpus.tsv`, el corpus crudo de la Práctica II (recuperación de artículos con modelo vectorial).

**Objetivo:** reunir ~500 artículos (125 por categoría) con sus metadatos, guardando la respuesta original de la API para trazabilidad.

Además de ejecutar la recolección, el notebook documenta **de dónde sale cada dato y qué se le hace en cada paso**, para poder estudiarlo. Cada sección explica qué hace la celda de código que la sigue, por qué, y cómo comprobarlo a mano.

## Recorrido de los datos

```
API de arXiv (export.arxiv.org/api/query)
   │  1 petición HTTP por página de 100 resultados, 4 categorías     → secciones 1-3
   ▼
data/raw/{categoria}_{inicio}.xml        ← respuesta original, sin tocar (10 archivos)
   │  parse_feed(): XML Atom → diccionarios, limpieza de espacios     → sección 4
   ▼
filtro: categoría primaria + sin duplicados + sin vacíos, 125/categoría → sección 5
   │
   ▼
data/scientific_raw_corpus.tsv           ← 500 filas, 10 columnas    → secciones 6-7
   │  verificación contra el XML crudo y contra arXiv en vivo         → sección 9
   ▼
(Fase 2: 02_normalizacion_corpus.ipynb → scientific_normalized_corpus.tsv)

Subproductos: logs/acquisition.log, results/corpus_distribution.md
```

## La fuente: API oficial de arXiv

arXiv es un repositorio abierto de artículos científicos (preprints). Ofrece una API pública y gratuita en `https://export.arxiv.org/api/query` que devuelve los metadatos en formato **Atom XML** (un estándar de feeds, parecido a RSS).

- Es oficial y gratuita: no hace falta registro ni clave.
- Devuelve metadatos ya estructurados (título, autores, resumen, categorías, fechas), así que no hay que hacer *scraping* de HTML.
- Las categorías de arXiv sirven de etiqueta temática para cada documento.

**Lo que no se descarga:** ni el PDF ni el texto completo. El sistema solo usa **título + abstract**.

**Normas de uso que se respetan** ([términos de uso](https://info.arxiv.org/help/api/tos.html)): como máximo una petición cada 3 s y un `User-Agent` que identifique al cliente.

## Decisiones de diseño

| Decisión | Valor | Motivo |
|---|---|---|
| Fuente | API oficial de arXiv (`export.arxiv.org/api/query`) | Gratuita, estable y con metadatos estructurados (Atom XML) |
| Categorías | `cs.CL`, `cs.CV`, `cs.IR`, `cs.LG` | Temas cercanos con solapamiento útil para consultas difíciles |
| Cuota | 125 documentos por categoría | Corpus equilibrado de 500 documentos |
| Asignación de categoría | **Categoría primaria** del artículo | Un artículo puede estar en varias categorías; usar la primaria evita duplicados y da una etiqueta única |
| Ventana temporal | 2026-01-01 a 2026-09-30 (corte fijo) | Hace la recolección **reproducible**: la misma consulta devuelve los mismos artículos |
| Orden | `submittedDate` descendente | Se toman los artículos más recientes antes del corte |
| Cortesía con la API | Pausa de 3 s entre peticiones, `User-Agent` identificable, reintentos con backoff | Recomendaciones de uso de arXiv |
| Caché | Si la página cruda ya existe en `data/raw/`, se reutiliza | Re-ejecutar el notebook no vuelve a consultar la API |

## 1. Configuración

Todos los parámetros de la recolección están en esta celda; cambiar uno de ellos cambia el corpus.

| Parámetro | Valor | Significado |
|---|---|---|
| `API_URL` | `https://export.arxiv.org/api/query` | Punto de acceso de la API |
| `CATEGORIES` | `cs.CL, cs.CV, cs.IR, cs.LG` | Computation and Language, Computer Vision, Information Retrieval, Machine Learning ([taxonomía](https://arxiv.org/category_taxonomy)) |
| `PER_CATEGORY` | 125 | 4 × 125 = 500 documentos |
| `DATE_FROM` / `DATE_TO` | `202601010000` / `202609302359` | Ventana fija, formato `YYYYMMDDHHMM` en GMT |
| `PAGE_SIZE` | 100 | Resultados por petición |
| `DELAY_S` | 3.0 | Segundos de espera tras cada petición (norma de arXiv) |
| `MAX_RETRIES` | 5 | Reintentos si falla una petición |
| `MAX_PAGES` | 30 | Tope de seguridad para no pedir páginas sin fin |
| `USER_AGENT` | `PracticaML-VSM/0.1 (...)` | Identifica al cliente ante arXiv |

### Las categorías

En arXiv, cada código tiene la forma `archivo.subcategoría`: `cs` es *Computer Science* (Informática) y lo que va detrás del punto es el área. Las cuatro elegidas son:

| Código | Nombre oficial (inglés) | Significado | Qué temas incluye |
|---|---|---|---|
| `cs.CL` | Computation and Language | Computación y Lenguaje | Procesamiento de lenguaje natural (PLN) y lingüística computacional: modelos de lenguaje (LLM), traducción automática, análisis de sentimiento, clasificación de texto, resumen automático |
| `cs.CV` | Computer Vision and Pattern Recognition | Visión por Computadora y Reconocimiento de Patrones | Procesamiento de imágenes y vídeo: detección y segmentación de objetos, reconocimiento, reconstrucción 3D, generación de imágenes |
| `cs.IR` | Information Retrieval | Recuperación de Información | Indexación, motores de búsqueda, ranking, sistemas de recomendación, evaluación de búsqueda (el tema de esta práctica) |
| `cs.LG` | Machine Learning | Aprendizaje Automático | Investigación general en aprendizaje automático: supervisado y no supervisado, aprendizaje por refuerzo, optimización, robustez, equidad (*fairness*), explicabilidad |

Los artículos también pueden tener **categorías secundarias** (*cross-listing*). En el corpus, las más frecuentes son:

| Código | Nombre oficial (inglés) | Significado | Apariciones como secundaria |
|---|---|---|---:|
| `cs.AI` | Artificial Intelligence | Inteligencia Artificial | 134 |
| `cs.LG` | Machine Learning | Aprendizaje Automático | 69 |
| `cs.CL` | Computation and Language | Computación y Lenguaje | 36 |
| `stat.ML` | Machine Learning (Statistics) | Aprendizaje Automático (desde la Estadística) | 16 |
| `cs.CV` | Computer Vision and Pattern Recognition | Visión por Computadora | 16 |
| `cs.RO` | Robotics | Robótica | 11 |
| `cs.MM` | Multimedia | Multimedia | 7 |
| `cs.MA` | Multiagent Systems | Sistemas Multiagente | 6 |
| `cs.IR` | Information Retrieval | Recuperación de Información | 6 |
| `cs.SD` | Sound | Sonido (audio y voz) | 5 |
| `math.OC` | Optimization and Control | Optimización y Control | 5 |
| `cs.HC` | Human-Computer Interaction | Interacción Persona-Computadora | 3 |

Fuente: [taxonomía oficial de arXiv](https://arxiv.org/category_taxonomy). Las secundarias solo se guardan en la columna `categories`; para asignar la etiqueta del documento se usa siempre la primaria (sección 5).

**Por qué un corte fijo:** si se pidiera "lo más reciente" sin límite, cada ejecución devolvería artículos distintos. Con `DATE_TO` fijo la consulta es reproducible.

`ROOT` se calcula para que el notebook funcione tanto si se abre desde `notebooks/` como desde la raíz del proyecto.

In [1]:
import csv
import logging
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# Raíz del proyecto (el notebook vive en notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

API_URL = "https://export.arxiv.org/api/query"
CATEGORIES = ["cs.CL", "cs.CV", "cs.IR", "cs.LG"]
PER_CATEGORY = 125
DATE_FROM = "202601010000"   # formato arXiv: YYYYMMDDHHMM
DATE_TO = "202609302359"     # fecha de corte fija
PAGE_SIZE = 100
DELAY_S = 3.0
MAX_RETRIES = 5
MAX_PAGES = 30               # tope de seguridad por categoría
USER_AGENT = "PracticaML-VSM/0.1 (proyecto academico de recuperacion de informacion)"

RAW_DIR = ROOT / "data" / "raw"
LOG_DIR = ROOT / "logs"
CORPUS_PATH = ROOT / "data" / "scientific_raw_corpus.tsv"
REPORT_PATH = ROOT / "results" / "corpus_distribution.md"

for d in (RAW_DIR, LOG_DIR, REPORT_PATH.parent):
    d.mkdir(parents=True, exist_ok=True)

ROOT

PosixPath('/home/mrangello/Documentos/PracticaML')

## 2. Registro de la adquisición

Todos los eventos (peticiones, reintentos, artículos descartados) se escriben en `logs/acquisition.log`, como pide el plan de trabajo. El log se **añade** en cada ejecución, no se sobrescribe, así que queda el historial completo.

Cómo leerlo:
- `=== Inicio de la recolección ... ===` marca cada ejecución del notebook.
- `OK cs.CL start=0 intento=1` significa que esa página se descargó **de la API** en el primer intento.
- Si en una ejecución no aparecen líneas `OK`, todas las páginas salieron de la caché de `data/raw/` (es lo que pasa en las ejecuciones de las 15:46 y 16:12 del log actual).
- `WARNING Error ...` indica un reintento; `ERROR Fallo definitivo` indica que se agotaron los reintentos.

In [2]:
logger = logging.getLogger("acquisition")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_DIR / "acquisition.log", encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
logger.info("=== Inicio de la recolección: categorías=%s cuota=%d ventana=%s-%s ===",
            CATEGORIES, PER_CATEGORY, DATE_FROM, DATE_TO)

## 3. Descarga de una página de resultados

### La consulta

`build_query` arma, para cada categoría, una consulta en el lenguaje de búsqueda de arXiv:

```
cat:cs.CL AND submittedDate:[202601010000 TO 202609302359]
```

- `cat:cs.CL` → artículos que tienen `cs.CL` **entre sus categorías**, sea como primaria o como secundaria (*cross-listing*). Por eso en la sección 5 hay que filtrar.
- `submittedDate:[... TO ...]` → enviados dentro de la ventana.

### Los parámetros de la URL

| Parámetro | Valor | Para qué |
|---|---|---|
| `search_query` | la consulta de arriba | Qué buscar |
| `start` | 0, 100, 200… | Desde qué resultado empezar (paginación) |
| `max_results` | 100 | Cuántos devolver |
| `sortBy` | `submittedDate` | Ordenar por fecha de envío |
| `sortOrder` | `descending` | Del más reciente al más antiguo |

**Pruébalo a mano:** esta URL en el navegador devuelve el mismo XML que está guardado en `data/raw/cs.CL_00000.xml` (salvo cambios posteriores en arXiv):

```
https://export.arxiv.org/api/query?search_query=cat:cs.CL+AND+submittedDate:[202601010000+TO+202609302359]&start=0&max_results=100&sortBy=submittedDate&sortOrder=descending
```

### Qué hace `fetch_page`, en orden

1. **Caché:** si ya existe `data/raw/{categoria}_{start:05d}.xml`, lo lee del disco y no llama a la API.
2. Si no existe, hace la petición HTTP con el `User-Agent`.
3. Espera 3 s (norma de arXiv).
4. Comprueba que la página no venga vacía por error: arXiv a veces devuelve 0 entradas de forma transitoria aunque `totalResults` diga que hay más. En ese caso lo trata como error.
5. Guarda los bytes **tal cual** en `data/raw/`. Es la prueba de lo que devolvió arXiv ese día; la sección 9 compara el TSV contra estos archivos.
6. Ante cualquier fallo (red, timeout, XML corrupto, página vacía) reintenta con *backoff* exponencial: espera 6 s, 12 s, 24 s… (`DELAY_S * 2 ** intento`). Tras 5 fallos lanza un error.

### Anatomía de la respuesta

Cabecera del feed (`data/raw/cs.CL_00000.xml`):

```xml
<opensearch:totalResults>21739</opensearch:totalResults>   ← artículos que cumplen la consulta
<opensearch:startIndex>0</opensearch:startIndex>
```

Cada artículo es un `<entry>`; a la derecha, la columna del TSV en la que acaba (sección 4):

```xml
<entry>
  <id>http://arxiv.org/abs/2610.00840v1</id>                     → arxiv_id (sin "v1")
  <title>Contextual trajectory and ...</title>                   → title
  <updated>2026-09-30T23:56:18Z</updated>                        (no se usa)
  <summary>Transformer-based large language models ...</summary> → abstract
  <category term="cs.CL" .../>                                   → categories
  <category term="cs.AI" .../>                                   → categories
  <published>2026-09-30T23:56:18Z</published>                    → date (YYYY-MM-DD)
  <arxiv:primary_category term="cs.CL"/>                         → primary_category
  <author><name>Grayson Wycliffe Storer</name></author>          → authors
  <author><name>Julia Witte Zimmerman</name></author>
  <arxiv:doi>...</arxiv:doi>                                     → doi (solo si existe)
</entry>
```

Las constantes `ATOM`, `ARXIV` y `OPENSEARCH` son los *namespaces* XML: `ElementTree` necesita el nombre completo (`{http://www.w3.org/2005/Atom}entry`) para encontrar cada etiqueta.

In [3]:
ATOM = "{http://www.w3.org/2005/Atom}"
ARXIV = "{http://arxiv.org/schemas/atom}"
OPENSEARCH = "{http://a9.com/-/spec/opensearch/1.1/}"


def build_query(category: str) -> str:
    return f"cat:{category} AND submittedDate:[{DATE_FROM} TO {DATE_TO}]"


def fetch_page(category: str, start: int) -> bytes:
    raw_path = RAW_DIR / f"{category}_{start:05d}.xml"
    if raw_path.exists():
        return raw_path.read_bytes()

    params = urllib.parse.urlencode({
        "search_query": build_query(category),
        "start": start,
        "max_results": PAGE_SIZE,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    })
    req = urllib.request.Request(f"{API_URL}?{params}", headers={"User-Agent": USER_AGENT})

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            time.sleep(DELAY_S)
            root = ET.fromstring(data)
            total = int(root.findtext(f"{OPENSEARCH}totalResults", "0"))
            if not root.findall(f"{ATOM}entry") and start < total:
                raise ValueError(f"página vacía (start={start}, total={total})")
            raw_path.write_bytes(data)
            logger.info("OK %s start=%d intento=%d", category, start, attempt)
            return data
        except (urllib.error.URLError, TimeoutError, ValueError, ET.ParseError) as e:
            wait = DELAY_S * 2 ** attempt
            logger.warning("Error %s start=%d intento=%d: %s; reintento en %.0fs",
                           category, start, attempt, e, wait)
            time.sleep(wait)
    logger.error("Fallo definitivo %s start=%d", category, start)
    raise RuntimeError(f"No se pudo descargar {category} start={start}")

## 4. Extracción y limpieza de campos

`parse_feed` recorre cada `<entry>` del XML y crea un diccionario por artículo:

| Columna | De dónde sale | Transformación |
|---|---|---|
| `arxiv_id` | `<id>` | `strip_version`: `http://arxiv.org/abs/2610.00840v1` → `2610.00840`. Es el `document_id` en todos los TSV |
| `doi` | `<arxiv:doi>` | Vacío si el autor no lo declaró (solo 14 de 500 lo tienen) |
| `title` | `<title>` | `clean()` |
| `authors` | todos los `<author><name>` | `clean()` y unidos con `"; "` |
| `abstract` | `<summary>` | `clean()` |
| `categories` | todos los `<category term>` | Unidos con espacio, p. ej. `cs.CL cs.AI` |
| `primary_category` | `<arxiv:primary_category term>` | Tal cual |
| `date` | `<published>` | Primeros 10 caracteres: fecha de la **primera versión** |
| `url` | se construye | `https://arxiv.org/abs/{arxiv_id}` |
| `retrieved_at` | fecha de modificación del XML crudo | Se asigna en la sección 5; es el momento real de la descarga |

**`clean()`** hace `" ".join(texto.split())`: parte el texto por cualquier espacio en blanco (espacios, tabuladores, saltos de línea) y lo une con un solo espacio. arXiv devuelve títulos y abstracts con saltos de línea, y en un TSV un tabulador o salto de línea dentro de un campo **rompe las columnas**.

**`strip_version()`** quita la URL y la versión (`v1`, `v2`…): el id sin versión identifica al artículo, no a una versión concreta. Usa `rpartition("v")` y solo corta si lo que sigue a la última `v` son dígitos.

In [4]:
def clean(text) -> str:
    return " ".join((text or "").split())


def strip_version(abs_url: str) -> str:
    ident = abs_url.split("/abs/")[-1]
    base, sep, ver = ident.rpartition("v")
    return base if sep and ver.isdigit() else ident


def parse_feed(data: bytes, retrieved_at: str) -> tuple[list[dict], int]:
    root = ET.fromstring(data)
    total = int(root.findtext(f"{OPENSEARCH}totalResults", "0"))
    records = []
    for e in root.findall(f"{ATOM}entry"):
        arxiv_id = strip_version(e.findtext(f"{ATOM}id", ""))
        primary = e.find(f"{ARXIV}primary_category")
        records.append({
            "arxiv_id": arxiv_id,
            "doi": clean(e.findtext(f"{ARXIV}doi", "")),
            "title": clean(e.findtext(f"{ATOM}title")),
            "authors": "; ".join(clean(a.findtext(f"{ATOM}name")) for a in e.findall(f"{ATOM}author")),
            "abstract": clean(e.findtext(f"{ATOM}summary")),
            "categories": " ".join(c.get("term") for c in e.findall(f"{ATOM}category")),
            "primary_category": primary.get("term") if primary is not None else "",
            "date": (e.findtext(f"{ATOM}published") or "")[:10],
            "url": f"https://arxiv.org/abs/{arxiv_id}",
            "retrieved_at": retrieved_at,
        })
    return records, total

## 5. Recolección por categoría

`collect_category` recorre páginas de 100 resultados y aplica a cada entrada estas reglas, **en este orden**:

1. **¿Su categoría primaria es la que se busca?** Si no → descarte `otra_primaria`. La consulta `cat:cs.IR` también devuelve artículos que son, por ejemplo, de `cs.CL` y solo están *cross-listed* en `cs.IR`. Quedarse con la primaria da a cada documento **una única etiqueta**.
2. **¿Ya está en el corpus?** (`arxiv_id` en `seen`) Si sí → descarte `duplicado`. Con la regla 1 no debería ocurrir nunca, pero queda como protección.
3. **¿Tiene título y abstract?** Si no → descarte `vacio` (y aviso en el log).
4. Si pasa todo → se conserva. Al llegar a 125 se para.

Si una página no basta para llegar a 125, se pide la siguiente (`start += 100`). También se para si se agotan los resultados (`start >= total`) o se alcanza `MAX_PAGES`.

`retrieved_at` se toma de la fecha de modificación del XML crudo, así que no cambia al reutilizar la caché: refleja cuándo se descargó de verdad.

Las columnas de la tabla resultante: `vistos` (entradas revisadas), los tres motivos de descarte, `conservados` y `total_api` (el `totalResults` de arXiv para esa consulta). Siempre se cumple `vistos = otra_primaria + duplicado + vacio + conservados`.

In [5]:
def collect_category(category: str, seen: set[str]) -> tuple[list[dict], dict]:
    kept, stats = [], {"vistos": 0, "otra_primaria": 0, "duplicado": 0, "vacio": 0}
    start, total = 0, None
    while len(kept) < PER_CATEGORY and start // PAGE_SIZE < MAX_PAGES:
        data = fetch_page(category, start)
        # Momento real de descarga: fecha del XML crudo (no cambia al reutilizar la caché)
        mtime = (RAW_DIR / f"{category}_{start:05d}.xml").stat().st_mtime
        retrieved_at = datetime.fromtimestamp(mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        records, total = parse_feed(data, retrieved_at)
        if not records:
            break
        for r in records:
            stats["vistos"] += 1
            if r["primary_category"] != category:
                stats["otra_primaria"] += 1
            elif r["arxiv_id"] in seen:
                stats["duplicado"] += 1
            elif not r["title"] or not r["abstract"]:
                stats["vacio"] += 1
                logger.warning("Abstract/título vacío: %s", r["arxiv_id"])
            else:
                seen.add(r["arxiv_id"])
                kept.append(r)
                if len(kept) == PER_CATEGORY:
                    break
        start += PAGE_SIZE
        if start >= total:
            break
    stats.update(conservados=len(kept), total_api=total)
    logger.info("%s: %s", category, stats)
    if len(kept) < PER_CATEGORY:
        logger.warning("%s: solo %d de %d documentos", category, len(kept), PER_CATEGORY)
    return kept, stats


seen: set[str] = set()
all_records, all_stats = [], {}
for cat in CATEGORIES:
    recs, stats = collect_category(cat, seen)
    all_records.extend(recs)
    all_stats[cat] = stats
    print(f"{cat}: {stats}")

pd.DataFrame(all_stats).T

cs.CL: {'vistos': 182, 'otra_primaria': 57, 'duplicado': 0, 'vacio': 0, 'conservados': 125, 'total_api': 21739}
cs.CV: {'vistos': 150, 'otra_primaria': 25, 'duplicado': 0, 'vacio': 0, 'conservados': 125, 'total_api': 29710}
cs.IR: {'vistos': 224, 'otra_primaria': 99, 'duplicado': 0, 'vacio': 0, 'conservados': 125, 'total_api': 4113}
cs.LG: {'vistos': 209, 'otra_primaria': 84, 'duplicado': 0, 'vacio': 0, 'conservados': 125, 'total_api': 42587}


,vistos,otra_primaria,duplicado,vacio,conservados,total_api
cs.CL,182,57,0,0,125,21739
cs.CV,150,25,0,0,125,29710
cs.IR,224,99,0,0,125,4113
cs.LG,209,84,0,0,125,42587


### Cómo leer este resultado

- **Por qué cs.IR y cs.LG necesitan 3 páginas:** la consulta `cat:X` devuelve también los artículos donde `X` es solo categoría secundaria, y esos se descartan (`otra_primaria`). Cuantos más descartes, más páginas hacen falta para llegar a 125:

  | Categoría | vistos | otra_primaria | conservados | páginas |
  |---|---|---|---|---|
  | cs.CV | 150 | 25 | 125 | 2 |
  | cs.CL | 182 | 57 | 125 | 2 |
  | cs.LG | 209 | 84 | 125 | 3 |
  | cs.IR | 224 | 99 | 125 | 3 |

  cs.LG es la categoría "comodín": muchos artículos de visión, NLP o estadística se publican también en cs.LG sin que sea su primaria. En las dos primeras páginas solo había 116 válidos, así que hizo falta una tercera. cs.CV casi siempre es primaria y con 150 vistos ya llega a la cuota. La tercera página no se lee entera: el bucle se corta al llegar a 125 (por eso `vistos` no es múltiplo de 100).
- **Qué significan `_00000`, `_00100`, `_00200` en `data/raw/`:** son el parámetro `start` de cada página (`f"{category}_{start:05d}.xml"`), no un contador de archivos. `cs.LG_00200.xml` contiene los resultados 200 a 299 de la consulta de cs.LG. El relleno con ceros a 5 dígitos hace que se ordenen bien alfabéticamente. Por eso CL y CV tienen dos archivos (`00000`, `00100`) e IR y LG tienen tres.
- **0 duplicados y 0 vacíos:** el filtro por categoría primaria ya impide que un artículo entre dos veces.
- **`total_api`** solo dice cuántos artículos hay en la ventana; únicamente se revisan los necesarios para llegar a la cuota.

**Consecuencia: sesgo temporal.** Como se ordena del más reciente al más antiguo y solo se toman 125, el corpus no cubre de enero a septiembre, sino los últimos días antes del corte:

| Categoría | Fechas cubiertas |
|---|---|
| cs.CL | 2026-09-29 a 2026-09-30 |
| cs.CV | 2026-09-30 |
| cs.IR | 2026-09-17 a 2026-09-30 (categoría con menos volumen) |
| cs.LG | 2026-09-30 |

Conviene mencionarlo en el reporte como limitación del corpus.

**¿Por qué hay ids `2610.xxxxx` con fecha de septiembre?** Los 4 primeros dígitos del id (`YYMM`) corresponden al ciclo de anuncio de arXiv, no a la fecha de envío. Un artículo enviado el 30 de septiembre a las 23:56 GMT queda después del último cierre de septiembre y recibe un id de octubre. En el corpus hay 377 ids `2609` y 123 ids `2610`; todos tienen `date` dentro de la ventana.

## 6. Construcción y validación del corpus

Criterios de aceptación de la Fase 1:
- sin duplicados de `arxiv_id`,
- sin títulos ni abstracts vacíos,
- sin tabuladores ni saltos de línea dentro de los campos.

In [6]:
COLUMNS = ["arxiv_id", "doi", "title", "authors", "abstract", "categories",
           "primary_category", "date", "url", "retrieved_at"]
corpus = pd.DataFrame(all_records, columns=COLUMNS)

assert corpus["arxiv_id"].is_unique, "Hay arxiv_id duplicados"
assert (corpus["title"].str.len() > 0).all(), "Hay títulos vacíos"
assert (corpus["abstract"].str.len() > 0).all(), "Hay abstracts vacíos"
assert not corpus.apply(lambda c: c.astype(str).str.contains(r"[\t\n\r]")).any().any(), \
    "Hay tabuladores o saltos de línea en los campos"

print(f"{len(corpus)} documentos válidos")
corpus.head()

500 documentos válidos


,arxiv_id,doi,title,authors,abstract,categories,primary_category,date,url,retrieved_at
0,2610.00840,,Contextual trajectory and incremental contextu...,Grayson Wycliffe Storer; Julia Witte Zimmerman,Transformer-based large language models (LLMs)...,cs.CL cs.AI,cs.CL,2026-09-30,https://arxiv.org/abs/2610.00840,2026-10-11T21:36:53Z
1,2610.00833,,VERITYGATE: A Four-Gate Schema-Level Faithfuln...,Sachin Gupta,Fluent LLM explanations may not follow the evi...,cs.CL cs.LG,cs.CL,2026-09-30,https://arxiv.org/abs/2610.00833,2026-10-11T21:36:53Z
2,2610.00827,,Verbalized and Internal Probabilities Are Coup...,Sinead Williamson; Jiaxuan Li; Nick Foti; Russ...,Large language models carry an internal notion...,cs.CL cs.AI,cs.CL,2026-09-30,https://arxiv.org/abs/2610.00827,2026-10-11T21:36:53Z
3,2610.06903,,Component and Dimension Sparsity in Transforme...,Vincent Siu; Glenn Grant-Richards; Vlad Pavlov...,Activation steering manipulates large language...,cs.CL cs.AI cs.LG,cs.CL,2026-09-30,https://arxiv.org/abs/2610.06903,2026-10-11T21:36:53Z
4,2610.00795,,Can large language models unlock discrete data...,Umair A. Zaidi; An-Lun Wu; Wei-Chun Lin; Thoma...,Objective: To assess the accuracy and efficien...,cs.CL,cs.CL,2026-09-30,https://arxiv.org/abs/2610.00795,2026-10-11T21:36:53Z


## 7. Guardado del TSV y verificación de carga

In [7]:
corpus.to_csv(CORPUS_PATH, sep="\t", index=False, quoting=csv.QUOTE_MINIMAL)

reloaded = pd.read_csv(CORPUS_PATH, sep="\t", dtype=str, keep_default_na=False)
assert reloaded.shape == corpus.shape
assert (reloaded["arxiv_id"] == corpus["arxiv_id"]).all()
logger.info("Corpus guardado: %s (%d filas)", CORPUS_PATH, len(reloaded))
print(CORPUS_PATH, reloaded.shape)

/home/mrangello/Documentos/PracticaML/data/scientific_raw_corpus.tsv (500, 10)


## 8. Reporte de distribución

Distribución por categoría primaria, por mes de publicación y por fuente, más estadísticas de longitud de título y abstract (útiles para la sección *Corpus* del reporte). Se guarda también en `results/corpus_distribution.md`.

In [8]:
corpus["source"] = "arXiv API"
corpus["month"] = corpus["date"].str[:7]
corpus["n_cats"] = corpus["categories"].str.split().str.len()

by_cat = corpus["primary_category"].value_counts().rename("documentos").to_frame()
by_month = pd.crosstab(corpus["month"], corpus["primary_category"], margins=True, margins_name="Total")
by_source = corpus["source"].value_counts().rename("documentos").to_frame()
lengths = pd.DataFrame({
    "palabras_titulo": corpus["title"].str.split().str.len(),
    "palabras_abstract": corpus["abstract"].str.split().str.len(),
}).describe().round(1)
multi = corpus.groupby("primary_category")["n_cats"].apply(lambda s: (s > 1).mean()).round(3)

display(by_cat, by_month, by_source, lengths)
print("Proporción de artículos con más de una categoría (cross-listing):")
print(multi)

,documentos
primary_category,
cs.CL,125
cs.CV,125
cs.IR,125
cs.LG,125


primary_category,cs.CL,cs.CV,cs.IR,cs.LG,Total
month,,,,,
2026-09,125,125,125,125,500
Total,125,125,125,125,500


,documentos
source,
arXiv API,500


,palabras_titulo,palabras_abstract
count,500.0,500.0
mean,9.8,195.1
std,2.9,43.9
min,2.0,84.0
25%,8.0,165.0
50%,10.0,194.0
75%,12.0,226.0
max,20.0,326.0


Proporción de artículos con más de una categoría (cross-listing):
primary_category
cs.CL    0.560
cs.CV    0.432
cs.IR    0.432
cs.LG    0.592
Name: n_cats, dtype: float64


In [9]:
report = f"""# Distribución del corpus crudo

- Generado: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}
- Consulta: `cat:<categoria> AND submittedDate:[{DATE_FROM} TO {DATE_TO}]`, orden `submittedDate` descendente
- Criterio: categoría primaria, {PER_CATEGORY} documentos por categoría
- Total: {len(corpus)} documentos, rango de fechas {corpus['date'].min()} a {corpus['date'].max()}

## Por categoría primaria
{by_cat.to_markdown()}

## Por mes de publicación
{by_month.to_markdown()}

## Por fuente
{by_source.to_markdown()}

## Longitud (palabras)
{lengths.to_markdown()}

## Proporción con cross-listing
{multi.to_frame('proporcion').to_markdown()}

## Estadísticas de la recolección
{pd.DataFrame(all_stats).T.to_markdown()}
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)

/home/mrangello/Documentos/PracticaML/results/corpus_distribution.md


## 9. Verificación de fuentes (trazabilidad)

Se comprueba que el TSV refleja fielmente la fuente, en dos niveles:

1. **Contra el XML crudo** (`data/raw/`): para **todos** los documentos, `title`, `abstract`, `primary_category` y `date` del TSV deben ser idénticos a los del `<entry>` correspondiente. Detecta errores de extracción o de guardado del TSV.
2. **Contra arXiv en vivo**: para una muestra aleatoria (`random_state=42`, siempre la misma), se vuelve a pedir cada artículo a la API por su `arxiv_id` (`id_list`) y se compara. Detecta si el XML guardado ya no coincide con lo que publica arXiv hoy.

**Cómo verificarlo a mano:** abre cualquier `url` del TSV (p. ej. https://arxiv.org/abs/2610.00840) y compara título y resumen.

In [10]:
# 9.1 Índice arxiv_id -> (archivo crudo, entrada) a partir de todos los XML guardados
raw_index = {}
for xml_path in sorted(RAW_DIR.glob("*.xml")):
    recs, _ = parse_feed(xml_path.read_bytes(), "")
    for r in recs:
        raw_index.setdefault(r["arxiv_id"], (xml_path.name, r))

mismatches = []
for row in reloaded.itertuples():
    fname, raw = raw_index[row.arxiv_id]
    for field in ("title", "abstract", "primary_category", "date"):
        if getattr(row, field) != raw[field]:
            mismatches.append((row.arxiv_id, field, fname))

print(f"Documentos comparados contra el XML crudo: {len(reloaded)}")
print(f"Diferencias encontradas: {len(mismatches)}")
assert not mismatches, mismatches[:5]

reloaded.assign(archivo_crudo=reloaded["arxiv_id"].map(lambda i: raw_index[i][0]))[
    ["arxiv_id", "primary_category", "archivo_crudo", "url"]].head(5)


Documentos comparados contra el XML crudo: 500
Diferencias encontradas: 0


,arxiv_id,primary_category,archivo_crudo,url
0,2610.00840,cs.CL,cs.CL_00000.xml,https://arxiv.org/abs/2610.00840
1,2610.00833,cs.CL,cs.CL_00000.xml,https://arxiv.org/abs/2610.00833
2,2610.00827,cs.CL,cs.CL_00000.xml,https://arxiv.org/abs/2610.00827
3,2610.06903,cs.CL,cs.CL_00000.xml,https://arxiv.org/abs/2610.06903
4,2610.00795,cs.CL,cs.CL_00000.xml,https://arxiv.org/abs/2610.00795


In [11]:
# 9.2 Comparación en vivo con arXiv para una muestra aleatoria (una sola petición a la API)
VERIFY_ONLINE = True
SAMPLE_SIZE = 10

sample = reloaded.sample(SAMPLE_SIZE, random_state=42)
if VERIFY_ONLINE:
    params = urllib.parse.urlencode({"id_list": ",".join(sample["arxiv_id"]), "max_results": SAMPLE_SIZE})
    req = urllib.request.Request(f"{API_URL}?{params}", headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as resp:
        live = {r["arxiv_id"]: r for r in parse_feed(resp.read(), "")[0]}
    time.sleep(DELAY_S)

    check = sample[["arxiv_id", "title", "abstract", "url"]].copy()
    check["titulo_igual"] = check.apply(lambda r: r["title"] == live.get(r["arxiv_id"], {}).get("title"), axis=1)
    check["abstract_igual"] = check.apply(lambda r: r["abstract"] == live.get(r["arxiv_id"], {}).get("abstract"), axis=1)
    logger.info("Verificación en vivo: %d/%d títulos y %d/%d abstracts iguales",
                check["titulo_igual"].sum(), SAMPLE_SIZE, check["abstract_igual"].sum(), SAMPLE_SIZE)
    # Una diferencia aquí no es un error de extracción: puede significar que el autor subió una versión nueva
    display(check[["arxiv_id", "titulo_igual", "abstract_igual", "url"]])


,arxiv_id,titulo_igual,abstract_igual,url
361,2609.23111,True,True,https://arxiv.org/abs/2609.23111
73,2609.39385,True,True,https://arxiv.org/abs/2609.39385
374,2609.20050,True,True,https://arxiv.org/abs/2609.20050
155,2609.40317,True,True,https://arxiv.org/abs/2609.40317
104,2609.38851,True,True,https://arxiv.org/abs/2609.38851
394,2610.00707,True,True,https://arxiv.org/abs/2610.00707
377,2610.00831,True,True,https://arxiv.org/abs/2610.00831
124,2609.38510,True,True,https://arxiv.org/abs/2609.38510
68,2609.39514,True,True,https://arxiv.org/abs/2609.39514
450,2609.40149,True,True,https://arxiv.org/abs/2609.40149


**Resultado de la verificación:** 0 diferencias en los 500 documentos contra el XML crudo y 10/10 títulos y abstracts iguales contra arXiv en vivo (queda registrado en la última línea de `logs/acquisition.log`). Si algún día la comprobación en vivo fallara, no sería necesariamente un error de extracción: el autor pudo subir una versión nueva.

## Referencias

- arXiv API User's Manual: https://info.arxiv.org/help/api/user-manual.html
- arXiv API Terms of Use (límite de una petición cada 3 s): https://info.arxiv.org/help/api/tos.html
- Taxonomía de categorías de arXiv: https://arxiv.org/category_taxonomy
- Formato de los identificadores de arXiv: https://info.arxiv.org/help/arxiv_identifier.html

## 10. Resultado

Archivos generados:
- `data/raw/*.xml`: respuestas crudas de la API (trazabilidad y caché).
- `data/scientific_raw_corpus.tsv`: corpus crudo, `document_id = arxiv_id`.
- `logs/acquisition.log`: registro de peticiones, reintentos y descartes.
- `results/corpus_distribution.md`: reporte de distribución.

**Siguiente paso:** Fase 2, normalización con spaCy (`title_norm`, `abstract_norm`) en `02_normalizacion_corpus.ipynb`.

**Nota sobre la ventana temporal:** al ordenar por fecha descendente, los 125 artículos de cada categoría corresponden a los últimos días antes del corte. Si se quiere más diversidad temporal, se puede ampliar el muestreo (por ejemplo, repartir la cuota entre varios meses) cambiando la configuración de la sección 1.

## Cómo reproducir la recolección

1. Para forzar una descarga nueva, borra `data/raw/*.xml`. Si no, se usa la caché y el resultado es idéntico.
2. Ejecuta el notebook de principio a fin (≈1 minuto por las pausas de 3 s).
3. Revisa `logs/acquisition.log` y `results/corpus_distribution.md`.

Una descarga nueva podría diferir ligeramente (artículos retirados o con versiones nuevas); por eso los XML de `data/raw/` se guardan en el repositorio como copia de referencia.

**Archivos `*-checkpoint.*` en `.ipynb_checkpoints/`:** no los genera este notebook. Los crea Jupyter al abrir un archivo (un XML, el TSV) desde su navegador de archivos, como copia de autoguardado. No afectan al corpus: `RAW_DIR.glob("*.xml")` no entra en subcarpetas. Se pueden borrar sin problema y conviene excluirlos con `.ipynb_checkpoints/` en `.gitignore`.

## Preguntas para repasar

1. ¿Por qué la consulta `cat:cs.IR` devuelve artículos que luego se descartan?
2. ¿Qué pasaría con el TSV si no se aplicara `clean()` al abstract?
3. ¿Por qué el corpus solo cubre los últimos días de septiembre si la ventana empieza en enero? ¿Cómo lo cambiarías?
4. ¿Qué garantiza que dos ejecuciones del notebook den el mismo corpus?
5. ¿Qué detecta la verificación contra el XML crudo que no detecta la verificación en vivo, y al revés?